# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below:

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

To run this notebook on Google Colab, install `evaluate`.

In [1]:
!pip install -q evaluate

## 15.10 Lab 6: Text Classification using Embeddings

In this lab, you'll use pretrained **sentence embeddings** as features for a multi-class text classifier. The suggested model is the `sentence-transformers/all-MiniLM-L6-v2`. Each news article will be transformed into one 384-dimensional embedding, and a small PyTorch linear classifier will learn to map those embeddings into the four AG News classes.

### 15.10.1 Dataset

First, load AG News dataset from the Hugging Face Hub (`fancyzhx/ag_news`), shuffle the splits, and select 20,000 training samples and 4,000 test samples.

In [2]:
from datasets import load_dataset, DatasetDict

raw_datasets = load_dataset("fancyzhx/ag_news")

train_ds = raw_datasets["train"].shuffle(seed=42).select(range(20_000))
test_ds = raw_datasets["test"].shuffle(seed=42).select(range(4_000))

datasets = DatasetDict({"train": train_ds, "test": test_ds})
label_names = datasets["train"].features["label"].names

datasets, label_names

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

(DatasetDict({
     train: Dataset({
         features: ['text', 'label'],
         num_rows: 20000
     })
     test: Dataset({
         features: ['text', 'label'],
         num_rows: 4000
     })
 }),
 ['World', 'Sports', 'Business', 'Sci/Tech'])

Then, inspect one training sample.

In [3]:
sample = datasets["train"][0]
sample

{'text': 'Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.',
 'label': 0}

### 15.10.3 Encoder and Tokenizer

We'd like to produce embeddings for each and every sentence in our dataset. So, load both an encoder-based model and its corresponding tokenizer.

In [4]:
from transformers import AutoModel, AutoTokenizer

repo_id = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(repo_id)
encoder = AutoModel.from_pretrained(repo_id)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Then, we send the model to the device and set it to evaluation mode, as we'll be using it as a feature extractor.

In [5]:
import torch

device = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
encoder.to(device)
encoder.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.

Let's inspect the shape of the output of our model, given a tokenized version of our sample:

In [15]:
sample_tokenized = tokenizer(sample['text'], return_tensors='pt')
token_ids = sample_tokenized['input_ids'].to(device)
attention_mask = sample_tokenized['attention_mask'].to(device)
output = encoder(token_ids, attention_mask)
output.last_hidden_state.shape

torch.Size([1, 35, 384])

Alright, one sentence, N tokens long, each token represented by an embedding of length 384. It is perfectly correct, but we're still one step short of our actual need: a single embedding for each sentence.

### 15.10.4 Mean Pooling

In order to obtain a single embedding representing each sentence, we can resort to mean pooling, that is, taking the average value across all tokens in the sentence (excluding the padding tokens, as indicated by the attention mask). Write a helper function that performs this step.

In [9]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)
    return pooled.clamp(min=1e-9)

Applying the `mean_pooling()` function to the output of our model should give us the desired shape.

In [16]:
pooled_output = mean_pooling(output, attention_mask)
pooled_output.shape

torch.Size([1, 384])

### 15.10.5 Encoding Batches

Our goal is to encode, that is, to produce normalized sentence embeddings for the whole dataset. So, write a function that loops over the dataset slicing mini-batches out of them, and then:
- tokenize these mini-batches;
- send the inputs (token IDs and attention mas) to the encoder;
- applies mean pooling to get one embedding vector for sentence
- normalizes the embeddings (as we did when using cosine similarity for comparison)

In [27]:
from tqdm.auto import tqdm
import torch.nn.functional as F

def encode_texts(model, tokenizer, texts, batch_size=128):
    all_embeddings = []

    device = next(iter(model.parameters())).device

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.inference_mode():
            model_output = model(**inputs)
            pooled_embeddings = mean_pooling(model_output, inputs["attention_mask"])
            normalized_embeddings = F.normalize(pooled_embeddings, p=2, dim=1)

        all_embeddings.append(normalized_embeddings.cpu())

    return torch.cat(all_embeddings)

Once the function is ready, let's try it out on a handful of texts.

In [28]:
sample_embeddings = encode_texts(encoder, tokenizer, datasets["train"]["text"][:4])
sample_embeddings.shape

  0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([4, 384])

Did you get a shape of (4, 384)? Great!

### 15.10.6 Precomputing the Features

Let's prepare the full dataset now: encode both splits, and convert the labels into tensors as well.

In [29]:
train_embeddings = encode_texts(encoder, tokenizer, datasets["train"]["text"])
test_embeddings = encode_texts(encoder, tokenizer, datasets["test"]["text"])

train_labels = torch.tensor(datasets["train"]["label"], dtype=torch.long)
test_labels = torch.tensor(datasets["test"]["label"], dtype=torch.long)

train_embeddings.shape, test_embeddings.shape

  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

(torch.Size([20000, 384]), torch.Size([4000, 384]))

Our samples are just regular tensors now. We can build a traditional `TensorDataset` using embeddings and labels.

In [ ]:
train_data = TensorDataset(train_embeddings, train_labels)
test_data = TensorDataset(test_embeddings, test_labels)

### 15.10.7 Data Loaders

We're still missing data loaders for both splits. Do not forget to shuffle the training set.

In [30]:
from torch.utils.data import DataLoader, TensorDataset

dataloaders = {}
dataloaders["train"] = DataLoader(train_data, batch_size=256, shuffle=True)
dataloaders["test"] = DataLoader(test_data, batch_size=512)

### 15.10.8 Linear Classifier

Now it is time to create and train the classifier. The minimum viable model is a single `Linear` layer that takes an embedding as input, and produce as many logits as there are classes in our AG News dataset. As long as these inputs and outputs are preserved, the classifier can be as complex as you'd like it to be. Don't forget it to send it to the device.

In [31]:
import torch.nn as nn

torch.manual_seed(42)
classifier = nn.Linear(train_embeddings.shape[1], len(label_names))
classifier.to(device)

Linear(in_features=384, out_features=4, bias=True)

### 15.10.9 Loss and Optimizer

Create an appropriate loss function for our multi-class classification task, and choose an optimizer and learning rate to use in training.

In [32]:
import torch.optim as optim

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=3e-3)

### 15.10.10 Training

Now, you may write the training loop. It is mostly the typical stuff we've done time and again, especially since the dataset is made of simple tensors only.

In [33]:
from tqdm.auto import tqdm

num_epochs = 20
epoch_losses = []

for epoch in range(num_epochs):
    classifier.train()
    running_loss = 0.0

    for embeddings, labels in dataloaders["train"]:
        embeddings = embeddings.to(device)
        labels = labels.to(device)

        predictions = classifier(embeddings)
        loss = loss_fn(predictions, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloaders["train"])
    epoch_losses.append(epoch_loss)
    print(f"Epoch {epoch + 1:02d} | loss: {epoch_loss:.4f}")

Epoch 01 | loss: 1.0680
Epoch 02 | loss: 0.6950
Epoch 03 | loss: 0.5427
Epoch 04 | loss: 0.4701
Epoch 05 | loss: 0.4279
Epoch 06 | loss: 0.4030
Epoch 07 | loss: 0.3846
Epoch 08 | loss: 0.3706
Epoch 09 | loss: 0.3634
Epoch 10 | loss: 0.3565
Epoch 11 | loss: 0.3470
Epoch 12 | loss: 0.3429
Epoch 13 | loss: 0.3407
Epoch 14 | loss: 0.3348
Epoch 15 | loss: 0.3309
Epoch 16 | loss: 0.3275
Epoch 17 | loss: 0.3256
Epoch 18 | loss: 0.3227
Epoch 19 | loss: 0.3203
Epoch 20 | loss: 0.3202


### 15.10.11 Evaluation

If training went well, the loss went down to about 0.3 or a bit more. But, how well is the model classifying sentences, really? Let's use Hugging Face's `evaluate` package to compute typical metrics such as accuracy, precision, recall, and F1-score over the test set. 

Write a function that takes a model (the classifier part), a data loader (that produces embeddings), and a list of `evaluate` metrics, and then:
- loops over the data loader;
- sends a mini-batch of embeddings to the classifier to get logits;
- find the predicted class;
- adds a mini-batch of predictions and labels to each metric;
- computes the metrics at the end.

Then, call the function using the trained classifier and test set.

In [54]:
import evaluate

metric1 = evaluate.load("accuracy")
metric2 = evaluate.load("precision")
metric3 = evaluate.load("recall")
metric4 = evaluate.load("f1")

def evaluate_metrics(model, dataloader, metrics):
    device = next(iter(model.parameters())).device

    for embeddings, labels in dataloader:
        model.eval()
        with torch.inference_mode():
            logits = model(embeddings.to(device))
            predictions = logits.argmax(dim=-1).cpu()

        for metric in metrics:
            metric.add_batch(predictions=predictions, references=labels)

    # We specify the average=None argument during compute for precision, recall, and f1
    result = []
    for metric in metrics:
        if metric.name in ["precision", "recall", "f1"]:
            result.append(metric.compute(average=None))
        else:
            result.append(metric.compute())
    return result

test_metrics = evaluate_metrics(classifier, dataloaders["test"], [metric1, metric2, metric3, metric4])
test_metrics

[{'accuracy': 0.88775},
 {'precision': array([0.89714868, 0.95965418, 0.8429003 , 0.84756098])},
 {'recall': array([0.88365095, 0.97558594, 0.83783784, 0.85102041])},
 {'f1': array([0.89034866, 0.96755448, 0.84036145, 0.84928717])}]

### 15.10.13 Classifying New Text

What if we want to classify brand new sentences? We cannot simply send them straight to the classifier, as it expects embeddings, not raw text. First, the text must be tokenized and encoded, and then the resulting embeddings are used as features to the classifier we trained. Write a function to perform this two-step process (tip: you can use the `encode_texts()` helper function we developed earlier).

In [67]:
def classify_texts(encoder, classifier, tokenizer, label_names, texts):
    device = next(iter(classifier.parameters())).device
    embeddings = encode_texts(encoder, tokenizer, texts).to(device)

    classifier.eval()
    with torch.inference_mode():
        logits = classifier(embeddings)
        predictions = logits.argmax(dim=1).cpu()

    return [label_names[idx.item()] for idx in predictions]

Finally, try classifying a few sentences. Their topics/labels are straightforward to guess. Did the model get them right too?

In [68]:
examples = [
  "The central bank raised interest rates after inflation remained above target.",
  "The team scored twice in the final minutes to win the championship.",
  "A new processor architecture promises faster performance for mobile devices.",
  "Leaders met to negotiate a ceasefire and discuss the border dispute.",
]

list(zip(examples, classify_texts(encoder, classifier, tokenizer, label_names, examples)))

  0%|          | 0/1 [00:00<?, ?it/s]

[('The central bank raised interest rates after inflation remained above target.',
  'Business'),
 ('The team scored twice in the final minutes to win the championship.',
  'Sports'),
 ('A new processor architecture promises faster performance for mobile devices.',
  'Sci/Tech'),
 ('Leaders met to negotiate a ceasefire and discuss the border dispute.',
  'World')]